# 🚀 Multilingual Video Summarization: Training & Evaluation Pipeline (Version 2 - Updated)

> **📌 Phiên bản 2 (v2 Update):** Nâng cấp mở rộng từ mô hình đơn ngữ tiếng Anh (v1 - CODE_MODEL.ipynb) thành hệ thống tóm tắt đa ngôn ngữ **mT5 (vi · en · fr · es)** kết hợp **Hierarchical Map-Reduce** và tích hợp **Whisper ASR**.

---

Whisper ASR → mT5 đa ngôn ngữ (vi · en · fr · es) → Hierarchical Map-Reduce → Tóm tắt kèm chương theo thời gian

## Tổng quan

Notebook huấn luyện và đánh giá mô hình tóm tắt phụ đề/transcript video:

- **Một model cho bốn ngôn ngữ**, điều khiển bằng prefix `summarize <lang>:`, fine-tune từ `csebuetnlp/mT5_multilingual_XLSum`.
- **Dữ liệu video (VT-SSum)** được cắt thành cửa sổ ≤ 280 từ có nhãn, khớp với cách chunking lúc suy luận.
- **Video dài** được xử lý bằng Map-Reduce phân cấp theo thời gian, cho ra tóm tắt tổng và các chương có mốc thời gian.
- **Đánh giá** trên tập test riêng từng nguồn, so với baseline Lead-3 và Zero-shot, có paired bootstrap, ablation và thí nghiệm chịu lỗi ASR.
- **Huấn luyện bền vững**: resume qua nhiều phiên Kaggle bằng Hugging Face Hub, Time-guard, NaN-guard, Config-guard.

```
 video / audio ─► Whisper ASR ─┐
 phụ đề .srt/.vtt ==──────────┼─► segments (start, end, text)
 văn bản thuần ==─────────────┘            │
                                            ▼
                         Chunking theo thời gian (≤ 400 token / đoạn)
                                            │  MAP:  mT5 "summarize <lang>: ..."
                                            ▼
                      Tóm tắt từng đoạn  ──►  Chương [mm:ss – mm:ss]
                                            │  REDUCE (đệ quy ≤ 3 vòng)
                                            ▼
                                  Tóm tắt cuối cùng
```

## Cách chạy trên Kaggle

1. Accelerator: GPU T4 x2. Internet: ON.
2. Add-ons → Secrets: thêm `HF_TOKEN` (token Hugging Face quyền Write).
3. Sửa `HUB_REPO` ở Bước 2. Đặt `SMOKE_TEST = True` để chạy thử, sau đó đổi lại `False` và Run All.
4. Nếu phiên bị ngắt, chạy lại toàn bộ notebook với cùng cấu hình để tiếp tục từ checkpoint trên Hub.

## Đầu ra

`results.csv`, `macro.csv`, `bootstrap.csv`, `faithfulness.csv`, `ablation.csv`, `robustness.csv`, `bertscore.csv`, các biểu đồ `.png` và model card `README.md`, lưu ở `/kaggle/working/ckpt` và đồng bộ lên Hugging Face Hub.

## Bước 1 — Cài đặt & khởi tạo môi trường

In [ ]:
# Bước 1: Cài đặt thư viện & khởi tạo
import subprocess, sys, time, os, json, glob, re, random, math, shutil, unicodedata
from pathlib import Path

START_TIME = time.time()   # mốc giờ để TimeGuard dừng an toàn trước giới hạn 12h của Kaggle

# Ghim transformers<5 để tránh xung đột API của Seq2SeqTrainer.
# (Bỏ qua việc cài đặt khi chạy thử nghiệm có CFG_OVERRIDE.)
if "CFG_OVERRIDE" not in globals():
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q", "-U",
        "transformers>=4.40.0,<5.0.0", "datasets", "accelerate",
        "sentencepiece", "rouge_score", "bert_score", "huggingface_hub",
    ], check=True)

import numpy as np, pandas as pd, torch
import transformers, datasets
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq, TrainerCallback,
)
from datasets import Dataset, DatasetDict, load_dataset, concatenate_datasets
from rouge_score import rouge_scorer

try:
    from IPython.display import display
except Exception:          # chạy ngoài Jupyter
    display = print

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RESULTS = {}               # gom kết quả các bước để sinh model card

print(f"PyTorch {torch.__version__} | Transformers {transformers.__version__} | Datasets {datasets.__version__}")
print(f"Thiết bị: {DEVICE} | Số GPU: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  [GPU {i}] {torch.cuda.get_device_name(i)}")

## Bước 2 — Cấu hình (chỉ cần sửa ở đây)

| Nhóm | Ý nghĩa |
|---|---|
| **Mô hình & lưu trữ** | Model nền, repo Hugging Face đích, thư mục output |
| **Lịch huấn luyện** | Số epoch, giờ tối đa mỗi phiên |
| **Siêu tham số** | Độ dài, batch, learning rate (FP32 + Adafactor để mT5 không bị NaN) |
| **Dữ liệu** | Nguồn, số mẫu mỗi nguồn, cửa sổ VT-SSum |
| **Đánh giá** | Bật/tắt từng thí nghiệm (bootstrap, ablation, robustness) |

Các tham số ảnh hưởng đến huấn luyện (epoch, batch, LR, nguồn dữ liệu…) được **khoá** giữa các phiên. Đổi giữa chừng sẽ bị chặn để tránh train lẫn cấu hình. Các cờ *đánh giá* thì đổi tự do.

In [ ]:
# Bước 2: Cấu hình
SMOKE_TEST = False   # True → chạy thử nhanh với dữ liệu thu nhỏ để kiểm tra luồng

CFG = dict(
    # ── Mô hình & nơi lưu ==───────────────────────────────────────────
    MODEL_NAME = "csebuetnlp/mT5_multilingual_XLSum",   # đã pre-train tóm tắt trên 45 ngôn ngữ
    HUB_REPO   = "minhunhooo/video-subtitle-summarization-mt5",   # ← ĐỔI thành repo của bạn
    OUT_DIR    = "/kaggle/working/ckpt",

    # ── Lịch huấn luyện ==─────────────────────────────────────────────
    TOTAL_EPOCHS     = 4,
    STOP_AFTER_EPOCH = None,   # vd 2 → dừng sau epoch 2 của phiên này (để chia nhiều phiên)
    MAX_HOURS        = 10.5,   # tự lưu & dừng trước giới hạn 12h của Kaggle

    # ── Siêu tham số ==────────────────────────────────────────────────
    MAX_INPUT    = 512,        # token đầu vào tối đa
    MAX_TARGET   = 128,        # token tóm tắt tối đa
    BATCH        = 4,          # batch / GPU
    GRAD_ACC     = 8,          # batch hiệu dụng = BATCH × GRAD_ACC × số GPU
    LR           = 3e-4,
    WARMUP_RATIO = 0.05,
    PRECISION    = "fp32",     # mT5 + fp16 dễ tràn số → NaN. Giữ fp32 (T4 không hỗ trợ bf16)
    GRAD_CKPT    = False,      # True nếu thiếu VRAM

    # ── Dữ liệu ==─────────────────────────────────────────────────────
    # kind: vietnews | xlsum | vtssum | jsonl (file riêng, mỗi dòng {"text":..,"summary":..})
    SOURCES = [
        dict(name="vietnews", kind="vietnews", lang="vi", train=16000, val=150, test=300),
        dict(name="xlsum-vi", kind="xlsum", hf="vietnamese", lang="vi", train=8000, val=150, test=300),
        dict(name="xlsum-en", kind="xlsum", hf="english",    lang="en", train=6000, val=150, test=300),
        dict(name="xlsum-fr", kind="xlsum", hf="french",     lang="fr", train=3000, val=100, test=200),
        dict(name="xlsum-es", kind="xlsum", hf="spanish",    lang="es", train=3000, val=100, test=200),
        dict(name="vt-ssum (video en)", kind="vtssum", lang="en", path="auto",
             train=8000, val=150, test=300,
             max_docs=dict(train=2500, validation=200, test=400)),   # số video đọc mỗi split
        # Dữ liệu video tiếng Việt tự thu (nếu có) → bỏ comment và thêm vào VIDEO_SOURCES:
        # dict(name="vi-video (tự thu)", kind="jsonl", lang="vi", path="/kaggle/input/<dataset>/vi_video.jsonl",
        #      train=4000, val=100, test=200),
    ],
    VIDEO_SOURCES       = ["vt-ssum (video en)"],
    VIDEO_UPSAMPLE      = 1,                                   # nhân bản dữ liệu video trong train
    VTSSUM_WINDOW_WORDS = 280,                                 # độ dài cửa sổ huấn luyện (từ)
    VTSSUM_GIT_URL      = "https://github.com/Dod-o/VT-SSum",  # tự clone nếu chưa gắn dataset

    # ── Đánh giá (đổi tự do, không bị khoá) ==──────────────────────────
    EVAL_SUBSET   = 500,    # số mẫu val dùng đánh giá nhanh mỗi epoch
    EVAL_ZEROSHOT = True,   # đo baseline Zero-shot trước khi fine-tune
    RUN_BERTSCORE = True,
    BOOTSTRAP_N   = 2000,   # số lần lấy mẫu lại cho paired bootstrap
    CHUNK_TOKENS  = 400,    # kích thước đoạn của Map-Reduce (< MAX_INPUT để chừa chỗ cho prefix)
    RUN_ABLATION  = True,   # Map-Reduce vs cắt cụt
    ABLATION_N    = 30,
    FULL_VIDEO_N  = 30,     # số transcript video đầy đủ dùng cho ablation
    RUN_ROBUSTNESS = True,  # đường cong chịu lỗi ASR (mô phỏng)
    ROBUST_N       = 60,
    ROBUST_RATES   = [0.0, 0.05, 0.10, 0.20, 0.30],
    ROBUST_SOURCES = ["vietnews", "xlsum-en", "vt-ssum (video en)"],
    PUSH_FINAL_TO_ROOT = True,   # đẩy model tốt nhất lên gốc repo để from_pretrained("user/repo") chạy được

    ALLOW_CFG_CHANGE = False,
    SEED = 42,
)

if SMOKE_TEST:
    print("SMOKE TEST: dữ liệu thu nhỏ, 1 epoch")
    for s in CFG["SOURCES"]:
        s.update(train=60, val=10, test=10)
        if s["kind"] == "vtssum":
            s["max_docs"] = dict(train=40, validation=10, test=10)
    CFG.update(TOTAL_EPOCHS=1, EVAL_SUBSET=20, MAX_INPUT=256, EVAL_ZEROSHOT=False, RUN_BERTSCORE=False,
               MAX_HOURS=1, BOOTSTRAP_N=200, ABLATION_N=3, FULL_VIDEO_N=3, ROBUST_N=5,
               ROBUST_RATES=[0.0, 0.2], PUSH_FINAL_TO_ROOT=False)

CFG.update(globals().get("CFG_OVERRIDE", {}))

random.seed(CFG["SEED"]); np.random.seed(CFG["SEED"]); torch.manual_seed(CFG["SEED"])

OUT = Path(CFG["OUT_DIR"]);  OUT.mkdir(parents=True, exist_ok=True)
BEST_DIR = OUT / "best";     BEST_DIR.mkdir(exist_ok=True)
HUB = CFG["HUB_REPO"] if CFG["HUB_REPO"] and not SMOKE_TEST else None

print(f"Model nền : {CFG['MODEL_NAME']}")
print(f"Hub repo  : {HUB or '(tắt — chỉ lưu local, sẽ mất khi hết phiên)'}")
print(f"Epoch     : {CFG['TOTAL_EPOCHS']} | Batch hiệu dụng ≈ {CFG['BATCH']}×{CFG['GRAD_ACC']}×{max(1, torch.cuda.device_count())}")

## Bước 3 — Đăng nhập Hugging Face & khôi phục trạng thái từ Hub

Nếu phiên trước đã đẩy checkpoint / model tốt nhất / baseline lên Hub, bước này tải chúng về để **train tiếp** thay vì bắt đầu lại. Token đọc từ **Kaggle Secrets** nên không lộ trong mã nguồn.

In [ ]:
# Bước 3: Đăng nhập Hub + khôi phục trạng thái từ phiên trước
from huggingface_hub import HfApi, login, snapshot_download, hf_hub_download

api = None
if HUB:
    try:
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        tok = os.environ.get("HF_TOKEN")

    if tok:
        login(token=tok)
        api = HfApi()
        api.create_repo(HUB, private=False, exist_ok=True)
        print(f"Đăng nhập thành công: {api.whoami()['name']} | Repo: {HUB}")
    else:
        print("Cảnh báo: Không thấy secret HF_TOKEN → chỉ lưu local (mất khi hết phiên).")
        print("   Cách thêm: Add-ons → Secrets → tên 'HF_TOKEN', giá trị là token Write của Hugging Face.")
        HUB = None

if api and HUB:
    try:
        snapshot_download(
            HUB, local_dir=str(OUT / "hub"),
            allow_patterns=["best/*", "last-checkpoint/*", "zeroshot_preds.json", "train_config.json"],
        )
    except Exception as e:
        print("Hub chưa có dữ liệu từ phiên trước (bình thường nếu đây là lần chạy đầu):", str(e)[:100])

    hub_dir = OUT / "hub"
    # Model tốt nhất của phiên trước → BEST_DIR (để đánh giá cuối không bị thiếu model)
    if (hub_dir / "best" / "config.json").exists() and not (BEST_DIR / "config.json").exists():
        shutil.copytree(hub_dir / "best", BEST_DIR, dirs_exist_ok=True)
        print("Đã khôi phục model tốt nhất từ Hub.")
    for f in ("zeroshot_preds.json", "train_config.json"):
        if (hub_dir / f).exists() and not (OUT / f).exists():
            shutil.copy(hub_dir / f, OUT / f)
            print(f"Đã khôi phục {f}")

## Bước 4 — Nạp & chuẩn bị dữ liệu đa nguồn

| Nguồn | Nội dung | Vai trò |
|---|---|---|
| **VietNews** | Báo tiếng Việt | Tiếng Việt chính |
| **XL-Sum** (vi/en/fr/es) | Tin BBC đa ngôn ngữ | Đa ngôn ngữ |
| **VT-SSum** | Phụ đề bài giảng video tiếng Anh | Miền video |

**Về VT-SSum:** mỗi video có transcript rất dài (trung vị ≈ 3.000 từ) và nhãn *câu nào thuộc bản tóm tắt*. Nếu nhét cả transcript vào 512 token, model chỉ thấy ~12% nội dung nhưng phải dự đoán câu tóm tắt rải khắp video. Notebook này **cắt video thành cửa sổ ≤ 280 từ**, mỗi cửa sổ kèm đúng các câu được gắn nhãn tóm tắt. Đây cũng chính là cách chunking dùng lúc suy luận (Map-Reduce), nên huấn luyện và suy luận khớp nhau.

Ngoài ra:
- Loại ví dụ quá ngắn; **chống rò rỉ** train↔test bằng cách loại bài train có 300 ký tự đầu trùng bài test.
- Mỗi nguồn nạp độc lập; nguồn nào lỗi sẽ báo rõ, riêng nguồn tiếng Việt thiếu thì dừng hẳn.

In [ ]:
# Bước 4: Nạp dữ liệu
def clean(t):
    """Gộp khoảng trắng/xuống dòng."""
    return re.sub(r"\s+", " ", str(t or "")).strip()


# ---------- Tiện ích chuẩn hoá cột & split cho dataset HF ----------
TEXT_COLS = ("text", "article", "content", "document", "body")
SUMM_COLS = ("summary", "abstract", "highlights")

def to_text_summary(ds):
    """Đưa dataset về đúng 2 cột: text, summary."""
    cols = ds.column_names
    tc = next((c for c in TEXT_COLS if c in cols), None)
    sc = next((c for c in SUMM_COLS if c in cols), None)
    if tc is None or sc is None:
        raise ValueError(f"Không nhận ra cột văn bản/tóm tắt trong {cols}")
    ds = ds.select_columns([tc, sc])
    if tc != "text":    ds = ds.rename_column(tc, "text")
    if sc != "summary": ds = ds.rename_column(sc, "summary")
    return ds

def normalize_splits(dd, seed):
    """Đảm bảo có đủ train / validation / test (tự tách nếu thiếu)."""
    alias = {"train": ("train",), "validation": ("validation", "val", "dev"), "test": ("test",)}
    out = {}
    for key, names in alias.items():
        for n in names:
            if n in dd:
                out[key] = to_text_summary(dd[n]); break
    if "train" not in out:
        raise ValueError("Dataset không có split train")
    if "test" not in out:
        sp = out["train"].train_test_split(test_size=0.02, seed=seed)
        out["train"], out["test"] = sp["train"], sp["test"]
    if "validation" not in out:
        sp = out["train"].train_test_split(test_size=0.02, seed=seed)
        out["train"], out["validation"] = sp["train"], sp["test"]
    return out


# ---------- XL-Sum: thử lần lượt 3 cách (tuỳ phiên bản `datasets`) ----------
def load_xlsum(lang):
    base = f"hf://datasets/csebuetnlp/xlsum@refs%2Fconvert%2Fparquet/{lang}"
    attempts = [
        ("parquet",     lambda: load_dataset("parquet", data_files={s: f"{base}/{s}/0000.parquet" for s in ("train", "validation", "test")})),
        ("hub-default", lambda: load_dataset("csebuetnlp/xlsum", lang)),
        ("remote-code", lambda: load_dataset("csebuetnlp/xlsum", lang, trust_remote_code=True)),
    ]
    errs = []
    for name, fn in attempts:
        try:
            return fn()
        except Exception as e:
            errs.append(f"{name}: {str(e)[:90]}")
    raise RuntimeError(f"Không tải được XL-Sum/{lang} → " + " | ".join(errs))


# ---------- VT-SSum: tự tìm / tự clone, rồi cắt thành cửa sổ có nhãn ----------
def find_vtssum_root():
    """Tìm thư mục có đủ 3 thư mục con train/dev/test trong /kaggle/input."""
    for root in glob.glob("/kaggle/input/**/", recursive=True):
        if all(os.path.isdir(os.path.join(root, s)) for s in ("train", "dev", "test")):
            return root
    return None

def ensure_vtssum(src):
    p = src.get("path", "auto")
    if p != "auto":
        return p if os.path.isdir(p) else None
    root = find_vtssum_root()
    if root:
        return root
    target = "/kaggle/working/VT-SSum"
    if not os.path.isdir(os.path.join(target, "train")):
        print(f"Chưa gắn dataset VT-SSum → clone từ {CFG['VTSSUM_GIT_URL']} (~1 GB, vài phút)...")
        try:
            r = subprocess.run(["git", "clone", "--depth", "1", CFG["VTSSUM_GIT_URL"], target],
                               capture_output=True, text=True, timeout=1200)
            if r.returncode != 0:
                print("  clone thất bại:", r.stderr[-200:]); return None
        except Exception as e:
            print("  clone thất bại:", e); return None
    return target if os.path.isdir(os.path.join(target, "train")) else None

def read_vtssum_doc(path):
    """Đọc 1 video → (danh sách câu, danh sách clip). Mỗi clip: cờ + câu + nhãn (1 = thuộc tóm tắt)."""
    d = json.load(open(path, encoding="utf-8"))
    sentences = [s for seg in d.get("segmentation", []) for s in (seg if isinstance(seg, list) else [seg]) if isinstance(s, str)]
    def clip_order(k):
        tail = k.split("_")[-1]
        return int(tail) if tail.isdigit() else 10**9
    clips = []
    for key in sorted(d.get("summarization", {}), key=clip_order):
        clip = d["summarization"][key]
        rows = [r for r in clip.get("summarization_data", []) if r.get("sent")]
        clips.append(dict(flag=bool(clip.get("is_summarization_sample")),
                          sents=[r["sent"].strip() for r in rows],
                          labels=[int(r.get("label", 0)) for r in rows]))
    return sentences, clips

def vtssum_windows(clips, max_words):
    """Gom các câu liên tiếp (chỉ trong clip dùng để tóm tắt) thành cửa sổ ≤ max_words từ.
    Mục tiêu của cửa sổ = các câu được gắn nhãn 1 trong đó. Bỏ cửa sổ không có câu tóm tắt."""
    wins, cur_s, cur_t, cur_w = [], [], [], 0
    def flush():
        nonlocal cur_s, cur_t, cur_w
        if cur_s and cur_t:
            wins.append({"text": " ".join(cur_s), "summary": " ".join(cur_t)})
        cur_s, cur_t, cur_w = [], [], 0
    for clip in clips:
        if not clip["flag"]:
            flush(); continue
        for sent, label in zip(clip["sents"], clip["labels"]):
            w = len(sent.split())
            if cur_w + w > max_words and cur_s:
                flush()
            cur_s.append(sent); cur_w += w
            if label == 1:
                cur_t.append(sent)
    flush()
    return wins

def read_vtssum_split(folder, max_docs, max_words):
    files = sorted(f for f in os.listdir(folder) if f.endswith(".json"))
    files = files[:max_docs] if max_docs else files
    wins, fulls = [], []
    for f in files:
        try:
            sentences, clips = read_vtssum_doc(os.path.join(folder, f))
        except Exception:
            continue
        wins += vtssum_windows(clips, max_words)
        full_sum = [s for c in clips if c["flag"] for s, l in zip(c["sents"], c["labels"]) if l == 1]
        if sentences and full_sum:
            fulls.append({"text": " ".join(sentences), "summary": " ".join(full_sum)})
    return wins, fulls


FULL_VIDEO_TEST = []   # transcript video đầy đủ + tóm tắt toàn video (dùng cho ablation Map-Reduce)
VT_STATS = {}

def load_source(src):
    k = src["kind"]
    if k == "xlsum":
        return normalize_splits(load_xlsum(src["hf"]), CFG["SEED"])

    if k == "vietnews":
        return normalize_splits(load_dataset("nam194/vietnews"), CFG["SEED"])

    if k == "vtssum":
        root = ensure_vtssum(src)
        if not root:
            print(f"[BỎ QUA] Không có VT-SSum cho '{src['name']}'.")
            return None
        md = src.get("max_docs", {})
        out, w_words, f_words = {}, [], []
        for split, folder in (("train", "train"), ("validation", "dev"), ("test", "test")):
            wins, fulls = read_vtssum_split(os.path.join(root, folder), md.get(split), CFG["VTSSUM_WINDOW_WORDS"])
            out[split] = Dataset.from_pandas(pd.DataFrame(wins, columns=["text", "summary"]), preserve_index=False)
            if split == "train":
                w_words = [len(w["text"].split()) for w in wins]
                f_words = [len(f["text"].split()) for f in fulls]
            if split == "test":
                FULL_VIDEO_TEST.extend(fulls[:CFG["FULL_VIDEO_N"]])
        if w_words and f_words:
            VT_STATS.update(full_median=int(np.median(f_words)), window_median=int(np.median(w_words)),
                            n_docs=len(f_words), n_windows=len(w_words))
        return out

    if k == "jsonl":
        if not os.path.isfile(src["path"]):
            print(f"[BỎ QUA] Không thấy file {src['path']}"); return None
        df = pd.read_json(src["path"], lines=True)[["text", "summary"]]
        df = df.sample(frac=1, random_state=CFG["SEED"]).reset_index(drop=True)
        n = len(df); a, b = int(n * 0.9), int(n * 0.95)
        mk = lambda x: Dataset.from_pandas(x.reset_index(drop=True), preserve_index=False)
        return {"train": mk(df[:a]), "validation": mk(df[a:b]), "test": mk(df[b:])}

    raise ValueError(f"Kiểu nguồn không hỗ trợ: {k}")


def is_valid(e):
    return len(clean(e["text"]).split()) >= 20 and len(clean(e["summary"]).split()) >= 3

def cap(d, n):
    d = d.shuffle(seed=CFG["SEED"])
    return d.select(range(min(n, len(d)))) if n else d


parts = {"train": [], "validation": [], "test": []}
stats, loaded_langs = [], set()

for src in CFG["SOURCES"]:
    try:
        d = load_source(src)
    except Exception as e:
        print(f"[LỖI NGUỒN] {src['name']}: {str(e)[:300]}")
        continue
    if d is None:
        continue
    d = {s: x.filter(is_valid) for s, x in d.items()}

    # Chống rò rỉ: loại mẫu train có 300 ký tự đầu trùng với mẫu test
    test_prefixes = {clean(t)[:300] for t in d["test"]["text"]}
    n_before = len(d["train"])
    d["train"] = d["train"].filter(lambda e: clean(e["text"])[:300] not in test_prefixes)
    n_leak = n_before - len(d["train"])

    n_up = CFG["VIDEO_UPSAMPLE"] if src["name"] in CFG["VIDEO_SOURCES"] else 1
    for split, key in (("train", "train"), ("validation", "val"), ("test", "test")):
        x = cap(d[split], src.get(key))
        x = x.add_column("lang", [src["lang"]] * len(x)).add_column("src", [src["name"]] * len(x))
        parts[split] += [x] * (n_up if split == "train" else 1)
    loaded_langs.add(src["lang"])
    stats.append((src["name"], src["lang"], len(d["train"]) + n_leak, n_leak,
                  len(parts["train"][-1]) * n_up, len(parts["validation"][-1]), len(parts["test"][-1])))

assert "vi" in loaded_langs, "Không nạp được nguồn tiếng Việt nào — dừng lại để tránh train sai mục tiêu. Xem các dòng [LỖI NGUỒN] ở trên."

raw = DatasetDict({s: concatenate_datasets(v) for s, v in parts.items()})
n_tr = len(raw["train"])

print("== THỐNG KÊ DỮ LIỆU ==")
display(pd.DataFrame(stats, columns=["Nguồn", "Ngôn ngữ", "Có sẵn", "Loại (rò rỉ)", "Train dùng", "Val", "Test"]))
print(f"Tổng: TRAIN = {n_tr:,} | VAL = {len(raw['validation']):,} | TEST = {len(raw['test']):,}")
if VT_STATS:
    cover = 100 * min(1.0, CFG["MAX_INPUT"] / 1.3 / max(VT_STATS["full_median"], 1))
    print(f"\nVT-SSum: transcript đầy đủ có trung vị {VT_STATS['full_median']:,} từ "
          f"(≈ {cover:.0f}% vừa trong {CFG['MAX_INPUT']} token); cửa sổ huấn luyện có trung vị {VT_STATS['window_median']} từ "
          f"({VT_STATS['n_windows']:,} cửa sổ từ {VT_STATS['n_docs']:,} video).")

## Bước 5 — Tokenization với prefix ngôn ngữ

Mỗi mẫu được viết thành `summarize <lang>: <văn bản>`. Một model duy nhất phục vụ cả 4 ngôn ngữ nhờ prefix này.

In [ ]:
# Bước 5: Tokenization
tokenizer = AutoTokenizer.from_pretrained(CFG["MODEL_NAME"])

def format_input(text, lang):
    return f"summarize {lang}: " + clean(text)

def preprocess(batch):
    x = tokenizer([format_input(t, l) for t, l in zip(batch["text"], batch["lang"])],
                  max_length=CFG["MAX_INPUT"], truncation=True)
    y = tokenizer(text_target=[clean(t) for t in batch["summary"]],
                  max_length=CFG["MAX_TARGET"], truncation=True)
    x["labels"] = y["input_ids"]
    return x

tok_ds = DatasetDict({
    s: raw[s].map(preprocess, batched=True, remove_columns=raw[s].column_names, desc=f"Tokenizing {s}")
    for s in ("train", "validation")
})
eval_ds = tok_ds["validation"].shuffle(seed=CFG["SEED"]).select(range(min(CFG["EVAL_SUBSET"], len(tok_ds["validation"]))))

lens = [len(x) for x in tok_ds["train"].select(range(min(2000, n_tr)))["input_ids"]]
print(f"Độ dài đầu vào (token): trung vị = {int(np.median(lens))} | "
      f"chạm trần {CFG['MAX_INPUT']} = {100 * np.mean([l >= CFG['MAX_INPUT'] for l in lens]):.1f}%")

## Bước 6 — Bộ công cụ đo lường

Gồm các thành phần dùng ở các bước đánh giá phía sau:

- **ROUGE giữ dấu tiếng Việt** (Unicode NFC). `rouge_score` mặc định xoá mọi ký tự ngoài `a-z0-9`, làm chữ có dấu bị vỡ thành từng mảnh.
- **Paired bootstrap:** kiểm định hệ thống A có thật sự hơn B không (khoảng tin cậy 95%), thay vì chỉ so hai số trung bình.
- **Proxy độ trung thực:** tỉ lệ *con số trong tóm tắt có xuất hiện trong văn bản gốc*, và tỉ lệ *từ mới* so với nguồn (mức độ abstractive). Đây chỉ là kiểm tra thô, **không** thay thế đánh giá sự thật.
- **Mô phỏng lỗi ASR** (rớt từ, mất dấu, lặp từ) cho thí nghiệm chịu lỗi.

In [ ]:
# Bước 6: Công cụ đo lường
class UnicodeTokenizer:
    """Tách từ theo Unicode NFC, giữ nguyên dấu tiếng Việt (thay cho tokenizer mặc định của rouge_score)."""
    def tokenize(self, text):
        norm = unicodedata.normalize("NFC", str(text or "")).lower()
        return re.sub(r"[^\w\s]", " ", norm).split()

ROUGE_KEYS = ("rouge1", "rouge2", "rougeL")
_rouge = rouge_scorer.RougeScorer(list(ROUGE_KEYS), tokenizer=UnicodeTokenizer())

def rouge_samples(preds, refs, field="fmeasure"):
    """ROUGE từng mẫu (thang 0–100). field: 'fmeasure' | 'precision' | 'recall'."""
    out = {k: [] for k in ROUGE_KEYS}
    for p, r in zip(preds, refs):
        s = _rouge.score(r, p)
        for k in ROUGE_KEYS:
            out[k].append(100.0 * getattr(s[k], field))
    return {k: np.array(v) for k, v in out.items()}

def rouge_mean(preds, refs, field="fmeasure"):
    return {k: round(float(v.mean()), 2) if len(v) else 0.0 for k, v in rouge_samples(preds, refs, field).items()}

def paired_bootstrap(a, b, n=2000, seed=0):
    """So sánh có cặp: trung bình (a − b), khoảng tin cậy 95% và xác suất cải thiện ≤ 0."""
    d = np.asarray(a, dtype=float) - np.asarray(b, dtype=float)
    rng = np.random.default_rng(seed)
    means = d[rng.integers(0, len(d), size=(n, len(d)))].mean(axis=1)
    lo, hi = np.percentile(means, [2.5, 97.5])
    return float(d.mean()), float(lo), float(hi), float((means <= 0).mean())

# ----- Proxy độ trung thực & mức abstractive -----
_NUM = re.compile(r"\d[\d.,]*\d|\d")
def _norm_num(s): return re.sub(r"[.,]", "", s)

def numeric_precision(source, summary):
    """Tỉ lệ con số trong tóm tắt xuất hiện trong nguồn (None nếu tóm tắt không có số)."""
    nums = {_norm_num(x) for x in _NUM.findall(summary)}
    if not nums:
        return None
    src_nums = {_norm_num(x) for x in _NUM.findall(source)}
    return len(nums & src_nums) / len(nums)

def novel_unigram_rate(source, summary):
    """Tỉ lệ từ trong tóm tắt không xuất hiện trong nguồn (cao = diễn đạt lại nhiều hơn)."""
    tk = UnicodeTokenizer().tokenize
    s_tokens, src_set = tk(summary), set(tk(source))
    return None if not s_tokens else sum(w not in src_set for w in s_tokens) / len(s_tokens)

def mean_ignore_none(xs):
    xs = [x for x in xs if x is not None]
    return float(np.mean(xs)) if xs else float("nan")

# ----- Lead-3 (baseline trích chọn) -----
def lead3(text):
    return " ".join(re.split(r"(?<=[.!?…])\s+", clean(text))[:3])

# ----- Mô phỏng lỗi ASR -----
def strip_marks(w):
    w = w.replace("đ", "d").replace("Đ", "D")
    return "".join(c for c in unicodedata.normalize("NFD", w) if unicodedata.category(c) != "Mn")

def add_asr_noise(text, rate, rng):
    """Mỗi từ có xác suất `rate` bị lỗi: rớt từ / mất dấu (hoặc đảo chữ) / lặp từ (chia đều 3 loại)."""
    out = []
    for w in text.split():
        r = rng.random()
        if r < rate / 3:                       # rớt từ
            continue
        elif r < 2 * rate / 3:                 # nghe nhầm: mất dấu, hoặc đảo 2 chữ giữa từ
            v = strip_marks(w)
            if v == w and len(w) > 3:
                i = int(rng.integers(1, len(w) - 2))
                v = w[:i] + w[i + 1] + w[i] + w[i + 2:]
            out.append(v)
        elif r < rate:                         # lặp từ
            out += [w, w]
        else:
            out.append(w)
    return " ".join(out)

# ----- Nhận diện ngôn ngữ nhẹ (cho suy luận khi không chỉ định lang) -----
_VI_CHARS = re.compile(r"[ăâđêôơưáàảãạấầẩẫậắằẳẵặéèẻẽẹếềểễệíìỉĩịóòỏõọốồổỗộớờởỡợúùủũụứừửữựýỳỷỹỵ]")
_STOP = {
    "fr": {"le", "la", "les", "des", "est", "une", "que", "pour", "dans", "et", "du", "au", "sur"},
    "es": {"el", "los", "las", "una", "que", "es", "para", "por", "con", "del", "y", "en"},
    "en": {"the", "and", "of", "to", "is", "in", "that", "for", "with", "on"},
}
def detect_lang(text):
    low = text.lower()
    letters = sum(c.isalpha() for c in low) or 1
    if len(_VI_CHARS.findall(low)) / letters > 0.03:
        return "vi"
    words = re.findall(r"\w+", low)
    scores = {l: sum(w in sw for w in words) for l, sw in _STOP.items()}
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else "en"

print("Bộ công cụ đo lường đã sẵn sàng.")

## Bước 7 — Engine suy luận: sinh văn bản & Map-Reduce phân cấp có chương

**Vấn đề:** mT5 chỉ đọc được ~512 token, nhưng video dài hàng nghìn từ.

**Giải pháp (Hierarchical Map-Reduce):**
1. **Chunking theo thời gian:** gom các segment liên tiếp thành đoạn ≤ 400 token, mỗi đoạn *giữ khoảng thời gian* `[start, end]` của nó.
2. **MAP:** tóm tắt từng đoạn → mỗi bản tóm tắt chính là một **chương** có mốc thời gian.
3. **REDUCE (đệ quy):** nối các tóm tắt rồi tóm tắt lại, tối đa 3 vòng, cho đến khi vừa trong một đoạn.

In [ ]:
# Bước 7: Engine suy luận
@torch.no_grad()
def generate(m, texts, langs, beams=4, bs=16, desc=None):
    """Sinh tóm tắt hàng loạt (sắp theo độ dài để giảm padding). Trả về list theo đúng thứ tự đầu vào."""
    m.eval()
    dev = next(m.parameters()).device
    order = np.argsort([len(t) for t in texts])[::-1]
    res = [None] * len(texts)
    starts = range(0, len(texts), bs)
    for i in (tqdm(starts, desc=desc, leave=False) if desc else starts):
        idx = order[i:i + bs]
        enc = tokenizer([format_input(texts[j], langs[j]) for j in idx], max_length=CFG["MAX_INPUT"],
                        truncation=True, padding=True, return_tensors="pt").to(dev)
        out = m.generate(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"],
                         max_new_tokens=CFG["MAX_TARGET"], num_beams=beams,
                         no_repeat_ngram_size=3, early_stopping=True)
        for j, s in zip(idx, tokenizer.batch_decode(out, skip_special_tokens=True)):
            res[j] = s.strip()
    return res


def n_tokens(texts):
    return [len(x) for x in tokenizer(list(texts), add_special_tokens=False)["input_ids"]]

def fmt_ts(sec):
    if sec is None: return "--:--"
    sec = int(sec); h, r = divmod(sec, 3600); m_, s = divmod(r, 60)
    return f"{h:02d}:{m_:02d}:{s:02d}" if h else f"{m_:02d}:{s:02d}"

def to_segments(text):
    """Văn bản thuần → danh sách segment (chưa có thời gian), tách theo câu."""
    sents = [s for s in re.split(r"(?<=[.!?…])\s+", clean(text)) if s]
    return [dict(start=None, end=None, text=s) for s in sents]

def pack_segments(segs, max_tokens):
    """Gom segment liên tiếp thành đoạn ≤ max_tokens, giữ khoảng thời gian [start, end] của đoạn."""
    if not segs:
        return []
    counts = n_tokens([s["text"] for s in segs])
    chunks, cur, n = [], [], 0
    def flush():
        nonlocal cur, n
        if cur:
            st = next((s["start"] for s in cur if s["start"] is not None), None)
            en = next((s["end"] for s in reversed(cur) if s["end"] is not None), None)
            chunks.append(dict(start=st, end=en, text=" ".join(s["text"] for s in cur)))
        cur, n = [], 0
    for s, k in zip(segs, counts):
        if k > max_tokens:                       # một câu quá dài (vd ASR không dấu câu) → cắt theo từ
            flush()
            words = s["text"].split()
            step = max(20, int(len(words) * max_tokens / k * 0.9))
            for i in range(0, len(words), step):
                chunks.append(dict(start=s["start"], end=s["end"], text=" ".join(words[i:i + step])))
            continue
        if n + k > max_tokens and cur:
            flush()
        cur.append(s); n += k
    flush()
    return chunks


def summarize_long(source, lang=None, model=None, beams=4, max_rounds=3):
    """Tóm tắt văn bản/video dài bằng Map-Reduce phân cấp.
    source: str (văn bản thuần) hoặc list[dict(start, end, text)] (phụ đề có thời gian).
    Trả về dict: summary, chapters[{start,end,summary}], n_chunks, rounds, input_tokens, lang."""
    model = model or eval_model
    segs = to_segments(source) if isinstance(source, str) else list(source)
    full = " ".join(s["text"] for s in segs)
    lang = lang or detect_lang(full)
    total = n_tokens([full])[0]
    L = CFG["CHUNK_TOKENS"]

    if total <= L:                                           # ngắn: tóm tắt trực tiếp
        return dict(summary=generate(model, [full], [lang], beams)[0], chapters=[], n_chunks=1,
                    rounds=0, input_tokens=total, lang=lang)

    chunks = pack_segments(segs, L)                          # ── MAP
    parts = generate(model, [c["text"] for c in chunks], [lang] * len(chunks), beams)
    chapters = [dict(start=c["start"], end=c["end"], summary=p) for c, p in zip(chunks, parts)]

    text, rounds = " ".join(parts), 1                        # ── REDUCE (đệ quy)
    while rounds < max_rounds and n_tokens([text])[0] > L:
        sub = pack_segments(to_segments(text), L)
        if len(sub) <= 1:
            break
        text = " ".join(generate(model, [c["text"] for c in sub], [lang] * len(sub), beams))
        rounds += 1
    final = generate(model, [text], [lang], beams)[0]
    return dict(summary=final, chapters=chapters, n_chunks=len(chunks), rounds=rounds,
                input_tokens=total, lang=lang)

print("Engine suy luận đã sẵn sàng (chunk = %d token)." % CFG["CHUNK_TOKENS"])

## Bước 8 — Model, Callbacks & Trainer

| Callback | Công dụng |
|---|---|
| **TimeGuard** | Hết `MAX_HOURS` thì lưu checkpoint và dừng êm, không để Kaggle cắt ngang |
| **NanGuard** | Loss NaN/Inf thì dừng ngay, khỏi tốn giờ GPU |
| **SpeedReport** | Sau 30 step in ra ước lượng thời gian mỗi epoch |
| **HubSync** | Mỗi epoch đẩy checkpoint lên Hub ở tiến trình nền (để resume khi rớt phiên) |
| **BestSaver** | Giữ model có **ROUGE-L val** cao nhất, đồng bộ lên Hub |

In [ ]:
# Bước 8: Model, callbacks, trainer
def compute_metrics(eval_pred):
    """Metric khi validate cuối mỗi epoch (ROUGE giữ dấu tiếng Việt)."""
    preds, labels = eval_pred
    if isinstance(preds, tuple):
        preds = preds[0]
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    p = [s.strip() for s in tokenizer.batch_decode(preds, skip_special_tokens=True)]
    l = [s.strip() for s in tokenizer.batch_decode(labels, skip_special_tokens=True)]
    m = rouge_mean(p, l)
    m["gen_len"] = float(np.mean([len(x.split()) for x in p]))
    return m

model = AutoModelForSeq2SeqLM.from_pretrained(CFG["MODEL_NAME"])
if CFG["GRAD_CKPT"]:
    model.gradient_checkpointing_enable(); model.config.use_cache = False
print(f"Số tham số: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M")

n_dev = max(1, torch.cuda.device_count())
steps_per_epoch = math.ceil(n_tr / (CFG["BATCH"] * CFG["GRAD_ACC"] * n_dev))
WARMUP_STEPS = max(1, int(CFG["WARMUP_RATIO"] * steps_per_epoch * CFG["TOTAL_EPOCHS"]))

class TimeGuard(TrainerCallback):
    def on_step_end(self, args, state, control, **kw):
        if (time.time() - START_TIME) / 3600.0 > CFG["MAX_HOURS"]:
            print(f"\n[TimeGuard] Đạt {CFG['MAX_HOURS']}h → lưu checkpoint và dừng an toàn.")
            control.should_save = True; control.should_training_stop = True
        return control
    def on_epoch_end(self, args, state, control, **kw):
        lim = CFG["STOP_AFTER_EPOCH"]
        if lim and state.epoch >= lim - 1e-3:
            print(f"\n[STOP] Xong epoch {lim} như đã chỉ định cho phiên này.")
            control.should_training_stop = True
        return control

class NanGuard(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kw):
        loss = (logs or {}).get("loss")
        if loss is not None and (math.isnan(loss) or math.isinf(loss)):
            print("\n[NanGuard] Loss NaN/Inf → dừng huấn luyện.")
            control.should_training_stop = True
        return control

class SpeedReport(TrainerCallback):
    def on_train_begin(self, args, state, control, **kw):
        self.t0 = None
    def on_step_end(self, args, state, control, **kw):
        if self.t0 is None:
            self.t0, self.s0 = time.time(), state.global_step
        elif state.global_step - self.s0 == 30:
            dt = (time.time() - self.t0) / 30.0
            print(f"\n[Tốc độ] ~{dt:.2f}s/step | ~{dt * steps_per_epoch / 3600:.1f}h/epoch | "
                  f"ước tính tổng ~{dt * steps_per_epoch * CFG['TOTAL_EPOCHS'] / 3600:.1f}h")
        return control

_uploads = []   # các lần upload nền lên Hub

class HubSync(TrainerCallback):
    def on_save(self, args, state, control, **kw):
        if not (api and HUB):
            return
        ck = Path(args.output_dir) / f"checkpoint-{state.global_step}"
        if ck.exists():
            _uploads.append(api.upload_folder(
                folder_path=str(ck), repo_id=HUB, path_in_repo="last-checkpoint", delete_patterns="*",
                commit_message=f"Checkpoint step {state.global_step} (epoch {state.epoch:.2f})", run_as_future=True))

class BestSaver(TrainerCallback):
    def __init__(self):
        self.best = -1.0
        f = BEST_DIR / "best.json"
        if f.exists():
            self.best = json.load(open(f)).get("rougeL", -1.0)
        print(f"ROUGE-L val tốt nhất hiện có: {self.best}")
    def on_evaluate(self, args, state, control, metrics=None, model=None, **kw):
        sc = (metrics or {}).get("eval_rougeL")
        if sc is not None and sc > self.best:
            self.best = sc
            m = model.module if hasattr(model, "module") else model
            m.save_pretrained(BEST_DIR); tokenizer.save_pretrained(BEST_DIR)
            json.dump({"rougeL": sc, "epoch": state.epoch, "step": state.global_step}, open(BEST_DIR / "best.json", "w"))
            print(f"\n[Best] Kỷ lục mới: ROUGE-L val = {sc} (epoch {state.epoch:.2f})")
            if api and HUB:
                _uploads.append(api.upload_folder(folder_path=str(BEST_DIR), repo_id=HUB, path_in_repo="best",
                                                  commit_message=f"Best ROUGE-L {sc}", run_as_future=True))

args_dict = dict(
    output_dir=str(OUT), num_train_epochs=CFG["TOTAL_EPOCHS"],
    per_device_train_batch_size=CFG["BATCH"], per_device_eval_batch_size=CFG["BATCH"] * 2,
    gradient_accumulation_steps=CFG["GRAD_ACC"], learning_rate=CFG["LR"], warmup_steps=WARMUP_STEPS,
    lr_scheduler_type="linear", optim="adafactor", weight_decay=0.0, max_grad_norm=1.0,
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=2,
    predict_with_generate=True, generation_max_length=CFG["MAX_TARGET"], generation_num_beams=1,
    bf16=(CFG["PRECISION"] == "bf16" and torch.cuda.is_available() and torch.cuda.is_bf16_supported()),
    fp16=(CFG["PRECISION"] == "fp16" and torch.cuda.is_available()),
    logging_steps=25, report_to="none", dataloader_num_workers=2, seed=CFG["SEED"],
    push_to_hub=False, load_best_model_at_end=False,
)
try:
    train_args = Seq2SeqTrainingArguments(**args_dict)
except TypeError:                                   # transformers cũ dùng tên evaluation_strategy
    args_dict["evaluation_strategy"] = args_dict.pop("eval_strategy")
    train_args = Seq2SeqTrainingArguments(**args_dict)

trainer_kw = dict(
    model=model, args=train_args, train_dataset=tok_ds["train"], eval_dataset=eval_ds,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model, label_pad_token_id=-100),
    compute_metrics=compute_metrics,
    callbacks=[TimeGuard(), NanGuard(), SpeedReport(), HubSync(), BestSaver()],
)
try:
    trainer = Seq2SeqTrainer(processing_class=tokenizer, **trainer_kw)
except TypeError:
    trainer = Seq2SeqTrainer(tokenizer=tokenizer, **trainer_kw)

print(f"~{steps_per_epoch} bước/epoch | tổng ~{steps_per_epoch * CFG['TOTAL_EPOCHS']} bước | warmup {WARMUP_STEPS} bước")

## Bước 9 — Baseline trước khi huấn luyện: Lead-3 và Zero-shot

Đo hai mốc so sánh trước khi fine-tune: **Zero-shot** (model gốc) và **Lead-3** (lấy 3 câu đầu). Dự đoán Zero-shot được lưu lại và đồng bộ lên Hub để dùng cho kiểm định thống kê ở Bước 13.

In [ ]:
# Bước 9: Baseline: Zero-shot & Lead-3
# Tập test riêng cho từng nguồn (giữ nguyên thứ tự để so sánh có cặp)
TEST = {}
_srcs = list(raw["test"]["src"])
for name in dict.fromkeys(_srcs):
    sub = raw["test"].select([i for i, s in enumerate(_srcs) if s == name])
    TEST[name] = dict(text=list(sub["text"]), ref=list(sub["summary"]), lang=list(sub["lang"]))

def predict_all(m, beams=4, desc="Sinh tóm tắt"):
    return {n: generate(m, t["text"], t["lang"], beams=beams, desc=f"{desc} · {n}") for n, t in TEST.items()}

ZS_PATH = OUT / "zeroshot_preds.json"
PRED_ZS = None
if CFG["EVAL_ZEROSHOT"]:
    has_resume = any(OUT.glob("checkpoint-*")) or (OUT / "hub" / "last-checkpoint").exists()
    if ZS_PATH.exists():
        PRED_ZS = json.load(open(ZS_PATH, encoding="utf-8"))
        print("Dùng lại dự đoán Zero-shot đã lưu.")
    elif not has_resume:
        model.to(DEVICE)
        print("Đang đo Zero-shot (model chưa fine-tune)...")
        PRED_ZS = predict_all(model, desc="Zero-shot")
        json.dump(PRED_ZS, open(ZS_PATH, "w", encoding="utf-8"), ensure_ascii=False)
        if api and HUB:
            api.upload_file(path_or_fileobj=str(ZS_PATH), path_in_repo="zeroshot_preds.json", repo_id=HUB)
    else:
        print("Cảnh báo: Đang resume nhưng không còn dự đoán Zero-shot → bỏ qua so sánh Zero-shot.")

if PRED_ZS:
    rows = []
    for n, t in TEST.items():
        if n in PRED_ZS and len(PRED_ZS[n]) == len(t["ref"]):
            rows.append({"Nguồn": n, "N": len(t["ref"]), "Hệ thống": "Lead-3", **rouge_mean([lead3(x) for x in t["text"]], t["ref"])})
            rows.append({"Nguồn": n, "N": len(t["ref"]), "Hệ thống": "Zero-shot", **rouge_mean(PRED_ZS[n], t["ref"])})
    print("== BASELINE TRƯỚC HUẤN LUYỆN (ROUGE F1, test độc lập) ==")
    display(pd.DataFrame(rows))

## Bước 10 — Phát hiện checkpoint & kiểm tra khớp cấu hình (Resume)

Tự tìm checkpoint mới nhất (local, Hub, hoặc dataset đã gắn vào) và tiếp tục từ đó. Trước khi nối, so sánh cấu hình với phiên trước; **khác thì dừng** để tránh train lẫn lộn hai cấu hình.

In [ ]:
# Bước 10: Resume & Config-guard
def get_step(p):
    try:
        return json.load(open(Path(p) / "trainer_state.json")).get("global_step", -1)
    except Exception:
        return -1

cands = glob.glob(str(OUT / "checkpoint-*"))
cands.append(str(OUT / "hub" / "last-checkpoint"))
cands += [str(Path(p).parent) for p in glob.glob("/kaggle/input/**/trainer_state.json", recursive=True)]
cands = [c for c in cands if get_step(c) >= 0]
resume_path = max(cands, key=get_step) if cands else None
print("→ Resume từ:", f"{resume_path} (step {get_step(resume_path)})" if resume_path else "train mới từ step 0")

GUARD_KEYS = ["MODEL_NAME", "TOTAL_EPOCHS", "BATCH", "GRAD_ACC", "LR", "WARMUP_RATIO", "MAX_INPUT", "MAX_TARGET",
              "SOURCES", "VIDEO_UPSAMPLE", "VTSSUM_WINDOW_WORDS", "SEED", "PRECISION"]
cfg_now = json.loads(json.dumps({k: CFG[k] for k in GUARD_KEYS}, sort_keys=True, default=str))
cfg_now["n_train"] = n_tr
cfg_now["steps_per_epoch"] = steps_per_epoch

cfg_file = OUT / "train_config.json"
if resume_path and cfg_file.exists() and not CFG["ALLOW_CFG_CHANGE"]:
    old = json.load(open(cfg_file))
    diff = [k for k in cfg_now if old.get(k) != cfg_now[k]]
    assert not diff, (f"Cấu hình khác phiên trước ở: {diff}. Hãy giữ nguyên cấu hình, "
                      f"hoặc đặt ALLOW_CFG_CHANGE=True nếu bạn chủ ý đổi.")
json.dump(cfg_now, open(cfg_file, "w"), indent=2, ensure_ascii=False)
if api and HUB:
    api.upload_file(path_or_fileobj=str(cfg_file), path_in_repo="train_config.json", repo_id=HUB)

## Bước 11 — Huấn luyện

Theo dõi dòng `[Tốc độ]` sau ~30 step đầu để biết tổng thời gian dự kiến. Nếu vượt `MAX_HOURS`, TimeGuard sẽ dừng êm và bạn chỉ cần chạy lại notebook ở phiên sau.

In [ ]:
# Bước 11: Huấn luyện
saved_ep = 0
if resume_path:
    try:
        saved_ep = json.load(open(Path(resume_path) / "trainer_state.json")).get("epoch", 0)
    except Exception:
        pass

if saved_ep >= CFG["TOTAL_EPOCHS"] - 1e-3:
    print(f"Đã đủ {CFG['TOTAL_EPOCHS']} epoch từ trước → bỏ qua huấn luyện.")
    FINAL_EPOCH = saved_ep
else:
    print(f"Bắt đầu huấn luyện (mục tiêu {CFG['TOTAL_EPOCHS']} epoch)...\n")
    trainer.train(resume_from_checkpoint=resume_path)
    FINAL_EPOCH = trainer.state.epoch

# Chờ các upload nền hoàn tất và báo lỗi nếu có (không để lỗi mạng im lặng)
for fut in _uploads:
    try:
        fut.result()
    except Exception as e:
        print("Cảnh báo: Upload lên Hub thất bại:", str(e)[:200])

print(f"\nKết thúc phiên: epoch {FINAL_EPOCH:.2f} / {CFG['TOTAL_EPOCHS']}")

## Bước 12 — Đường cong huấn luyện

In [ ]:
# Bước 12: Biểu đồ huấn luyện
v_logs = [l for l in trainer.state.log_history if "eval_rougeL" in l]
t_logs = [l for l in trainer.state.log_history if "loss" in l and "eval_loss" not in l]

if v_logs:
    fig, ax = plt.subplots(1, 3, figsize=(16, 4))
    ax[0].plot([l["step"] for l in t_logs], [l["loss"] for l in t_logs], color="#2563eb")
    ax[0].set(title="Train loss", xlabel="Step", ylabel="Loss")
    ax[1].plot([l["epoch"] for l in v_logs], [l["eval_loss"] for l in v_logs], "o-", color="#ea580c")
    ax[1].set(title="Validation loss", xlabel="Epoch", ylabel="Loss")
    for k, c in (("rouge1", "#16a34a"), ("rouge2", "#dc2626"), ("rougeL", "#7c3aed")):
        ax[2].plot([l["epoch"] for l in v_logs], [l[f"eval_{k}"] for l in v_logs], "o-", label=k, color=c)
    ax[2].set(title="ROUGE trên tập validation", xlabel="Epoch", ylabel="F1 (%)"); ax[2].legend()
    for a in ax:
        a.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(OUT / "training_curves.png", dpi=150); plt.show()
else:
    print("(Phiên này chưa có bước validate nào để vẽ.)")

## Bước 13 — Đánh giá cuối: test độc lập, so sánh hệ thống, kiểm định thống kê

Chỉ chạy khi đã đủ số epoch. Với mỗi nguồn dữ liệu (tập test không dùng để train):

1. **Bảng ROUGE** cho Lead-3 · Zero-shot · **Fine-tuned (ours)**, cùng điểm trung bình macro.
2. **Paired bootstrap:** mức cải thiện ROUGE-L của model so với từng baseline kèm khoảng tin cậy 95%. Nếu khoảng này **không chứa 0** thì cải thiện có ý nghĩa thống kê.
3. **BERTScore** đa ngữ (ngoài trùng lặp từ, đo độ gần về nghĩa).

In [ ]:
# Bước 13: Đánh giá cuối
FT, ZS, L3 = "Fine-tuned (ours)", "Zero-shot", "Lead-3"
finished = FINAL_EPOCH >= CFG["TOTAL_EPOCHS"] - 1e-3

if not finished:
    print(f"Model mới ở epoch {FINAL_EPOCH:.2f}/{CFG['TOTAL_EPOCHS']} → chạy lại notebook ở phiên sau để hoàn tất. "
          "Các bước 13–18 sẽ báo thiếu dữ liệu nếu chạy tiếp lúc này.")
else:
    if (BEST_DIR / "config.json").exists():
        eval_model = AutoModelForSeq2SeqLM.from_pretrained(BEST_DIR)
        print("Đánh giá model tốt nhất theo ROUGE-L val:", json.load(open(BEST_DIR / "best.json")))
    else:
        eval_model = trainer.model
        print("Cảnh báo: Không có thư mục best/ → đánh giá model ở cuối huấn luyện.")
    eval_model.to(DEVICE).eval()

    PRED_FT = predict_all(eval_model, desc="Fine-tuned")

    # --- Bảng ROUGE & điểm từng mẫu (để bootstrap) ---
    rows, PER = [], {}
    for n, t in TEST.items():
        systems = {L3: [lead3(x) for x in t["text"]]}
        if PRED_ZS and n in PRED_ZS and len(PRED_ZS[n]) == len(t["ref"]):
            systems[ZS] = PRED_ZS[n]
        systems[FT] = PRED_FT[n]
        for sysname, preds in systems.items():
            PER[(n, sysname)] = rouge_samples(preds, t["ref"])
            rows.append({"Nguồn": n, "N": len(t["ref"]), "Hệ thống": sysname,
                         **{k: round(float(v.mean()), 2) for k, v in PER[(n, sysname)].items()}})
    results = pd.DataFrame(rows)
    order = {L3: 0, ZS: 1, FT: 2}
    results = results.sort_values(["Nguồn", "Hệ thống"], key=lambda c: c.map(order) if c.name == "Hệ thống" else c, kind="stable").reset_index(drop=True)
    results.to_csv(OUT / "results.csv", index=False)
    print("== ROUGE F1 TRÊN TEST ĐỘC LẬP TỪNG NGUỒN ==")
    display(results)

    macro = results.groupby("Hệ thống")[list(ROUGE_KEYS)].mean().round(2).reindex([s for s in order if s in set(results["Hệ thống"])])
    macro.to_csv(OUT / "macro.csv")
    print("== TRUNG BÌNH MACRO (mỗi nguồn một phiếu) ==")
    display(macro)

    # --- Paired bootstrap ---
    boot_rows = []
    for n in TEST:
        for base in (L3, ZS):
            if (n, base) in PER:
                delta, lo, hi, p = paired_bootstrap(PER[(n, FT)]["rougeL"], PER[(n, base)]["rougeL"], CFG["BOOTSTRAP_N"], CFG["SEED"])
                boot_rows.append({"Nguồn": n, "So với": base, "ΔROUGE-L": round(delta, 2), "CI95 thấp": round(lo, 2),
                                  "CI95 cao": round(hi, 2), "p(Δ≤0)": round(p, 4), "Có ý nghĩa": bool(lo > 0)})
    boot = pd.DataFrame(boot_rows)
    boot.to_csv(OUT / "bootstrap.csv", index=False)
    print("== PAIRED BOOTSTRAP: Fine-tuned so với baseline (ROUGE-L) ==")
    display(boot)

    # --- BERTScore ---
    bert_df = None
    if CFG["RUN_BERTSCORE"]:
        try:
            from bert_score import score as bert_score_fn
            b_rows = []
            for n, t in TEST.items():
                k = min(200, len(t["ref"]))
                f1 = bert_score_fn(PRED_FT[n][:k], t["ref"][:k], model_type="bert-base-multilingual-cased",
                                   num_layers=9, verbose=False)[2]
                b_rows.append({"Nguồn": n, "BERTScore-F1": round(float(f1.mean()), 4)})
            bert_df = pd.DataFrame(b_rows); bert_df.to_csv(OUT / "bertscore.csv", index=False)
            print("== BERTSCORE ĐA NGỮ =="); display(bert_df)
        except Exception as e:
            print("Bỏ qua BERTScore:", str(e)[:120])

    # --- Biểu đồ ROUGE-L theo nguồn ---
    piv = results.pivot(index="Nguồn", columns="Hệ thống", values="rougeL")
    cols = [s for s in order if s in piv.columns]
    colors = {L3: "#9ca3af", ZS: "#f59e0b", FT: "#2563eb"}
    ax = piv[cols].plot.bar(figsize=(10, 4.5), rot=20, color=[colors[c] for c in cols])
    ax.set(title="ROUGE-L trên test độc lập từng nguồn", ylabel="ROUGE-L F1 (%)"); ax.grid(axis="y", alpha=0.3)
    plt.tight_layout(); plt.savefig(OUT / "rougeL_by_source.png", dpi=150); plt.show()

    RESULTS.update(results=results, macro=macro, boot=boot, bert=bert_df)

## Bước 14 — Phân tích độ trung thực & mức diễn đạt lại

ROUGE chỉ đo độ trùng lặp từ, **không đo tóm tắt có đúng sự thật hay không**. Hai chỉ số proxy dưới đây giúp phát hiện sớm hiện tượng "bịa":

- **Độ chính xác số liệu:** trong các con số xuất hiện ở bản tóm tắt, bao nhiêu % thật sự có trong văn bản gốc. Thấp = nghi ngờ bịa số.
- **Tỉ lệ từ mới:** bao nhiêu % từ trong tóm tắt không có trong nguồn. Cao = diễn đạt lại (abstractive); thấp = chép gần nguyên văn.

Đây chỉ là kiểm tra thô, không thay thế đánh giá sự thật bằng người chấm hoặc mô hình NLI.

In [ ]:
# Bước 14: Proxy độ trung thực
if not finished:
    print("Bỏ qua: chưa huấn luyện xong.")
else:
    f_rows = []
    for n, t in TEST.items():
        systems = {"Tham chiếu (reference)": t["ref"]}
        if PRED_ZS and n in PRED_ZS and len(PRED_ZS[n]) == len(t["ref"]):
            systems[ZS] = PRED_ZS[n]
        systems[FT] = PRED_FT[n]
        for sysname, preds in systems.items():
            nums = [numeric_precision(src, p) for src, p in zip(t["text"], preds)]
            f_rows.append({
                "Nguồn": n, "Hệ thống": sysname,
                "Số liệu có trong nguồn (%)": round(100 * mean_ignore_none(nums), 1),
                "% mẫu có chứa số": round(100 * np.mean([x is not None for x in nums]), 1),
                "Từ mới so với nguồn (%)": round(100 * mean_ignore_none([novel_unigram_rate(s, p) for s, p in zip(t["text"], preds)]), 1),
                "Độ dài TB (từ)": round(float(np.mean([len(p.split()) for p in preds])), 1),
            })
    faith = pd.DataFrame(f_rows)
    faith.to_csv(OUT / "faithfulness.csv", index=False)
    display(faith)
    RESULTS["faith"] = faith

## Bước 15 — Ablation: Map-Reduce phân cấp vs cắt cụt

Câu hỏi cốt lõi của đề tài: **chia nhỏ rồi tóm tắt phân cấp có tốt hơn cắt ở 512 token không?**

- **Cắt cụt (single-pass):** chỉ đọc 512 token đầu, bỏ phần còn lại.
- **Map-Reduce phân cấp:** đọc toàn bộ nội dung theo cách ở Bước 7.

Thí nghiệm chạy trên (a) transcript video VT-SSum *đầy đủ* với tóm tắt toàn video, và (b) các bài báo dài hơn 1,5 lần giới hạn đầu vào. Báo cáo cả **F1** và **Recall** vì tóm tắt tham chiếu của VT-SSum rất dài. Hiệu số có khoảng tin cậy bootstrap, nên kết quả nào cũng báo thật, kể cả khi không có cải thiện.

In [ ]:
# Bước 15: Ablation: Map-Reduce so với cắt cụt
abl_df = None
if not finished:
    print("Bỏ qua: chưa huấn luyện xong.")
elif not CFG["RUN_ABLATION"]:
    print("Bỏ qua: RUN_ABLATION = False.")
else:
    sets = {}
    if FULL_VIDEO_TEST:
        sets["Video đầy đủ (VT-SSum)"] = ([d["text"] for d in FULL_VIDEO_TEST], [d["summary"] for d in FULL_VIDEO_TEST],
                                           ["en"] * len(FULL_VIDEO_TEST))
    lt, lr_, ll = [], [], []
    for n, t in TEST.items():
        if n in CFG["VIDEO_SOURCES"]:
            continue
        long_idx = [i for i, k in enumerate(n_tokens(t["text"])) if k > 1.5 * CFG["MAX_INPUT"]][:CFG["ABLATION_N"]]
        lt += [t["text"][i] for i in long_idx]; lr_ += [t["ref"][i] for i in long_idx]; ll += [t["lang"][i] for i in long_idx]
    if lt:
        sets["Bài báo dài (> 1,5× giới hạn)"] = (lt[:CFG["ABLATION_N"]], lr_[:CFG["ABLATION_N"]], ll[:CFG["ABLATION_N"]])

    a_rows = []
    for name, (texts, refs, langs) in sets.items():
        trunc = generate(eval_model, texts, langs, desc=f"Cắt cụt · {name}")
        hier = [summarize_long(t, lang=l, model=eval_model)["summary"] for t, l in tqdm(list(zip(texts, langs)), desc=f"Map-Reduce · {name}", leave=False)]
        f_t, f_h = rouge_samples(trunc, refs), rouge_samples(hier, refs)
        r_t, r_h = rouge_samples(trunc, refs, "recall"), rouge_samples(hier, refs, "recall")
        d, lo, hi, p = paired_bootstrap(f_h["rougeL"], f_t["rougeL"], CFG["BOOTSTRAP_N"], CFG["SEED"])
        a_rows.append({
            "Tập": name, "N": len(texts), "Token TB": int(np.mean(n_tokens(texts))),
            "ROUGE-L F1 cắt cụt": round(float(f_t["rougeL"].mean()), 2), "ROUGE-L F1 Map-Reduce": round(float(f_h["rougeL"].mean()), 2),
            "ΔF1": round(d, 2), "CI95 thấp": round(lo, 2), "CI95 cao": round(hi, 2),
            "ROUGE-L Recall cắt cụt": round(float(r_t["rougeL"].mean()), 2), "ROUGE-L Recall Map-Reduce": round(float(r_h["rougeL"].mean()), 2),
            "Có ý nghĩa": bool(lo > 0),
        })
    if a_rows:
        abl_df = pd.DataFrame(a_rows); abl_df.to_csv(OUT / "ablation.csv", index=False)
        display(abl_df)
        RESULTS["ablation"] = abl_df
    else:
        print("Không có mẫu đủ dài để chạy ablation.")

## Bước 16 — Độ chịu lỗi ASR (mô phỏng)

Trong thực tế transcript đến từ Whisper nên có lỗi: rớt từ, nghe nhầm (mất dấu tiếng Việt), lặp từ. Thí nghiệm này **cố ý làm hỏng đầu vào** với tỉ lệ lỗi từ 0% đến 30% rồi xem ROUGE tụt bao nhiêu. Đường cong càng phẳng thì hệ thống càng bền.

Lưu ý: đây là lỗi mô phỏng, chưa phải đo trên đầu ra Whisper thật.

In [ ]:
# Bước 16: Đường cong chịu lỗi asr
rob_df = None
if not finished:
    print("Bỏ qua: chưa huấn luyện xong.")
elif not CFG["RUN_ROBUSTNESS"]:
    print("Bỏ qua: RUN_ROBUSTNESS = False.")
else:
    names = [n for n in CFG["ROBUST_SOURCES"] if n in TEST]
    if not names:
        print("Không có nguồn nào trong ROBUST_SOURCES.")
    else:
        rob_rows = []
        for n in names:
            t = TEST[n]
            k = min(CFG["ROBUST_N"], len(t["ref"]))
            idx = np.random.default_rng(CFG["SEED"]).permutation(len(t["ref"]))[:k]
            for rate in CFG["ROBUST_RATES"]:
                rng = np.random.default_rng(CFG["SEED"] + int(rate * 1000))   # cùng nhiễu cho mọi lần chạy
                noisy = [add_asr_noise(t["text"][i], rate, rng) for i in idx]
                preds = generate(eval_model, noisy, [t["lang"][i] for i in idx], beams=2, desc=f"{n} · lỗi {int(rate*100)}%")
                rob_rows.append({"Nguồn": n, "Tỉ lệ lỗi từ": rate, "N": k,
                                 **rouge_mean(preds, [t["ref"][i] for i in idx])})
        rob_df = pd.DataFrame(rob_rows)
        clean_L = rob_df[rob_df["Tỉ lệ lỗi từ"] == 0].set_index("Nguồn")["rougeL"]
        rob_df["% ROUGE-L còn lại"] = [round(100 * r["rougeL"] / clean_L[r["Nguồn"]], 1) if clean_L.get(r["Nguồn"], 0) else np.nan
                                       for _, r in rob_df.iterrows()]
        rob_df.to_csv(OUT / "robustness.csv", index=False)
        display(rob_df)

        fig, ax = plt.subplots(figsize=(7, 4))
        for n in names:
            s = rob_df[rob_df["Nguồn"] == n]
            ax.plot(100 * s["Tỉ lệ lỗi từ"], s["rougeL"], "o-", label=n)
        ax.set(title="Độ chịu lỗi ASR (lỗi mô phỏng)", xlabel="Tỉ lệ từ bị lỗi (%)", ylabel="ROUGE-L F1 (%)")
        ax.grid(alpha=0.3); ax.legend()
        plt.tight_layout(); plt.savefig(OUT / "robustness.png", dpi=150); plt.show()
        RESULTS["robust"] = rob_df

## Bước 17: Model card và đồng bộ Hub

Sinh `README.md` (model card) gồm cách dùng, kết quả đo được và các hạn chế, rồi đẩy kết quả cùng model tốt nhất lên gốc repo Hub để dùng trực tiếp bằng `from_pretrained("user/repo")`.

In [ ]:
# Bước 17: Model card + đồng bộ Hub
def to_md(df):
    head = "| " + " | ".join(map(str, df.columns)) + " |\n|" + "---|" * len(df.columns) + "\n"
    return head + "\n".join("| " + " | ".join(map(str, r)) + " |" for r in df.values)

if not finished:
    print("Bỏ qua: chưa huấn luyện xong.")
else:
    results, macro, boot = RESULTS["results"], RESULTS["macro"], RESULTS["boot"]
    sources_used = sorted(set(raw["train"]["src"]))
    card = """---
language: [vi, en, fr, es]
tags: [summarization, mt5, multilingual, video-transcript, vietnamese]
base_model: @@BASE@@
---
# Multilingual Video Transcript Summarizer (vi · en · fr · es)

Fine-tuned từ `@@BASE@@` trong @@EPOCHS@@ epoch trên @@NTR@@ mẫu từ: @@SRCS@@.

## Cách dùng
Đầu vào cần có prefix ngôn ngữ: `summarize <lang>: <văn bản>`, với `<lang>` ∈ {vi, en, fr, es}.
```python
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
tok = AutoTokenizer.from_pretrained("@@HUB@@")
model = AutoModelForSeq2SeqLM.from_pretrained("@@HUB@@")
x = tok("summarize vi: " + text, max_length=512, truncation=True, return_tensors="pt")
print(tok.decode(model.generate(**x, max_new_tokens=128, num_beams=4, no_repeat_ngram_size=3)[0], skip_special_tokens=True))
```
Với video/văn bản dài hơn 512 token, dùng Map-Reduce phân cấp (chunk ≤ @@CHUNK@@ token, tối đa 3 vòng reduce) thay vì cắt cụt.

## Kết quả (test độc lập từng nguồn, ROUGE F1 Unicode-NFC)
@@RESULTS@@

**Trung bình macro**

@@MACRO@@

**Kiểm định paired bootstrap (Fine-tuned so với baseline, ROUGE-L)**

@@BOOT@@
@@EXTRA@@
## Hạn chế
- Mô hình sinh có thể **bịa chi tiết**; ROUGE đo độ trùng lặp, không đo tính đúng sự thật.
- Tham chiếu của VT-SSum là *trích chọn câu*, khác với tóm tắt tóm lược của XL-Sum/VietNews, nên điểm giữa các nguồn không so sánh trực tiếp được.
- Thí nghiệm chịu lỗi ASR dùng lỗi **mô phỏng**, chưa phải đầu ra Whisper thật.
- Chỉ đánh giá vi/en/fr/es.
"""
    extra = ""
    if abl_df is not None and len(abl_df):
        extra += "\n**Ablation: Map-Reduce vs cắt cụt**\n\n" + to_md(abl_df) + "\n"
    if rob_df is not None and len(rob_df):
        extra += "\n**Độ chịu lỗi ASR (mô phỏng)**\n\n" + to_md(rob_df) + "\n"
    for k, v in {"@@BASE@@": CFG["MODEL_NAME"], "@@EPOCHS@@": str(CFG["TOTAL_EPOCHS"]), "@@NTR@@": f"{n_tr:,}",
                 "@@SRCS@@": ", ".join(sources_used), "@@HUB@@": HUB or "USER/REPO", "@@CHUNK@@": str(CFG["CHUNK_TOKENS"]),
                 "@@RESULTS@@": to_md(results), "@@MACRO@@": to_md(macro.reset_index()), "@@BOOT@@": to_md(boot),
                 "@@EXTRA@@": extra}.items():
        card = card.replace(k, v)
    open(OUT / "README.md", "w", encoding="utf-8").write(card)

    if api and HUB:
        files = ["README.md", "results.csv", "macro.csv", "bootstrap.csv", "faithfulness.csv", "ablation.csv",
                 "robustness.csv", "bertscore.csv", "rougeL_by_source.png", "training_curves.png", "robustness.png"]
        for f in files:
            if (OUT / f).exists():
                api.upload_file(path_or_fileobj=str(OUT / f), path_in_repo=f, repo_id=HUB)
        if CFG["PUSH_FINAL_TO_ROOT"]:
            api.upload_folder(folder_path=str(BEST_DIR), repo_id=HUB, ignore_patterns=["best.json"],
                              commit_message="Final best model at repo root")
        print(f"Đã đẩy model card, kết quả và model lên https://huggingface.co/{HUB}")

## Bước 18 — Demo: tóm tắt video dài kèm chương có mốc thời gian

Chạy engine ở Bước 7 trên một transcript dài thật từ tập test. Mốc thời gian ở đây là **mô phỏng** (giả định ~2,6 từ/giây) vì dữ liệu mẫu không có timestamp. Khi dùng Whisper thật, `start`/`end` lấy trực tiếp từ từng segment.

In [ ]:
# Bước 18: Demo Map-Reduce + chương
def with_fake_times(sentences, wps=2.6):
    """Gán mốc thời gian mô phỏng theo số từ (chỉ để minh hoạ)."""
    t, segs = 0.0, []
    for s in sentences:
        d = max(1.0, len(s.split()) / wps)
        segs.append(dict(start=t, end=t + d, text=s)); t += d
    return segs

def show(res):
    print(f"Ngôn ngữ: {res['lang']} | {res['input_tokens']:,} token | {res['n_chunks']} đoạn | {res['rounds']} vòng reduce\n")
    print(" TÓM TẮT\n" + res["summary"] + "\n")
    if res["chapters"]:
        print(" CHƯƠNG")
        for c in res["chapters"]:
            print(f"  [{fmt_ts(c['start'])} – {fmt_ts(c['end'])}] {c['summary']}")

if not finished:
    print("Bỏ qua: chưa huấn luyện xong.")
else:
    if FULL_VIDEO_TEST:
        demo_text, demo_lang, title = FULL_VIDEO_TEST[0]["text"], "en", "Video bài giảng (VT-SSum, tiếng Anh)"
    else:
        vi_src = next((n for n in TEST if TEST[n]["lang"][0] == "vi"), None)
        demo_text, demo_lang, title = " ".join(TEST[vi_src]["text"][:4]), "vi", f"4 bài ghép lại ({vi_src}, tiếng Việt)"
    print(f"=== DEMO: {title} ===")
    sents = [s for s in re.split(r"(?<=[.!?…])\s+", clean(demo_text)) if s]
    show(summarize_long(with_fake_times(sents), lang=demo_lang, model=eval_model))

## Bước tiếp theo

1. Kiểm tra `results.csv`, `bootstrap.csv`, `ablation.csv` và điền kết quả vào README của repo mã nguồn.
2. Dùng model trong ứng dụng/CLI của repo: `export VSUM_MODEL=<username>/<tên-repo>` rồi `python app.py`.
3. Mở rộng: thêm dữ liệu video tiếng Việt (nguồn `jsonl` ở Bước 2) và đo WER thật của Whisper trên đó.